# TT1 - MDM UBA - 2025

**Tariff classification using NLP**

New enviroment is needed for replication of doc2vec baseline

**doc2vec** & **fasttext** library

!pip install gensim==4.3.3

In [1]:
# Gral dependencies
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
from datetime import datetime
import re
from typing import Iterable, List, Tuple, Union, Optional, Dict, Any
from collections import defaultdict
import matplotlib.pyplot as plt

### Raw dataset

In [2]:
colspecs = [(0, 6), (6, None)]
data_type = {'HS06': str}
df = pd.read_fwf('data/raw_data_HScodes_desc.txt',
                 colspecs=colspecs, header=None,
                 names=['HS06', 'GOODS_DESCRIPTION'],
                 dtype=data_type)

### Quick EDA

null and duplicated samples

dropping duplicates

analyzing tops and bottoms regarding frequencies

In [3]:
# Quick EDA
print("=== Quick EDA ===")

# Add HS02 (chapter) and HS04 (heading)
df['HS04'] = df['HS06'].str[:4]
df['HS02'] = df['HS06'].str[:2]

print("Nulls per column:")
print(df.isnull().sum(), "")

print("Duplicate rows:", df.duplicated().sum(), "")

# Function to build and display freq tables
def freq_table(col, name):
    vc      = df[col].value_counts().rename('count')
    rel     = df[col].value_counts(normalize=True).rename('rel_freq')
    cum     = rel.cumsum().rename('cum_freq')
    summary = pd.concat([vc, rel, cum], axis=1)
    summary['rel_freq'] = (summary['rel_freq'] * 100).round(2).astype(str) + '%'
    summary['cum_freq'] = (summary['cum_freq'] * 100).round(2).astype(str) + '%'

    print(f"## Samples per {name} ({col})")
    print("### Top 10")
    print(summary.head(10).to_markdown(), "\n")
    print("### Bottom 10")
    print(summary.tail(10).to_markdown(), "\n")

# Dropping duplicates
df.drop_duplicates(inplace=True)

# Chapter-level (HS02)
freq_table('HS02', 'chapter')

# Heading-level (HS04)
freq_table('HS04', 'heading')

# Subheading-level (HS06)
freq_table('HS06', 'subheading')

=== Quick EDA ===
Nulls per column:
HS06                 0
GOODS_DESCRIPTION    0
HS04                 0
HS02                 0
dtype: int64 
Duplicate rows: 232220 
## Samples per chapter (HS02)
### Top 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     84 |   54901 | 20.5%      | 20.5%      |
|     85 |   33571 | 12.54%     | 33.04%     |
|     87 |   28476 | 10.63%     | 43.67%     |
|     73 |   16173 | 6.04%      | 49.71%     |
|     39 |   12218 | 4.56%      | 54.28%     |
|     90 |   11611 | 4.34%      | 58.61%     |
|     82 |    7972 | 2.98%      | 61.59%     |
|     94 |    7921 | 2.96%      | 64.55%     |
|     40 |    7526 | 2.81%      | 67.36%     |
|     83 |    4285 | 1.6%       | 68.96%     | 

### Bottom 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     41 |      22 | 0.01%      | 99.96%     |
|     81 |      19 | 0.01%      | 99.97%     |
|     45 |      19 | 0.01

### Preprocessing of text

In [4]:
stop_words = {'of', 'or', 'and', 'for', 'than', 'the', 'in', 'with', 'to', 'but', 'by'
             , 'whether', 'on', 'its', 'an', 'their', 'at', 'this', 'which', 'from'
             , 'as', 'be', 'is'}
alphabet_pattern = re.compile(r'[^a-zA-Z]')
alphabet_number_pattern = re.compile(r'[^a-zA-Z0-9]')
remove_pattern = re.compile(r'[\;\,\)\(\[\]\:]')


def refine_text_func(text):
    text = text.lower()
    text = ' '.join([w for w in text.split() if w not in stop_words])
    alphabet = re.sub(alphabet_pattern, ' ', text)
    alphabet_number = re.sub(alphabet_number_pattern, ' ', text)
    remove = re.sub(remove_pattern, ' ', text)
    result = ' '.join([text, alphabet, alphabet_number, remove])
    return result

In [5]:
df['PREPRO_DESCRIPTION'] = df['GOODS_DESCRIPTION'].progress_apply(lambda x: refine_text_func(x))

100%|██████████| 267780/267780 [00:01<00:00, 190012.56it/s]


### N-gram generation

In [6]:
def create_ngram_data(text, ngram_value=2):
    text_list = text.split()
    ngram_list = list(zip(*[text_list[i:] for i in range(ngram_value)]))
    result = []
    for n_data in ngram_list:
        result.append('_'.join(n_data))
    return ' '.join(result)

create_ngram_data('LIVE BREEDING FARM HORSE')

'LIVE_BREEDING BREEDING_FARM FARM_HORSE'

In [7]:
df['NGRAM_DESCRIPTION'] = df['PREPRO_DESCRIPTION'].progress_apply(lambda x: create_ngram_data(x))

100%|██████████| 267780/267780 [00:00<00:00, 271352.29it/s]


In [8]:
df.head()

,HS06,GOODS_DESCRIPTION,HS04,HS02,PREPRO_DESCRIPTION,NGRAM_DESCRIPTION
0,271019,BRAKE FLUID DOT 4 50X200ML,2710,27,brake fluid dot 4 50x200ml brake fluid dot ...,brake_fluid fluid_dot dot_4 4_50x200ml 50x200m...
1,847710,PLASTIC INJECTION MOULD MODEL 21A 110G DSM1010...,8477,84,plastic injection mould model 21a 110g dsm1010...,plastic_injection injection_mould mould_model ...
2,844399,LCD ASSEMBLY,8443,84,lcd assembly lcd assembly lcd assembly lcd ass...,lcd_assembly assembly_lcd lcd_assembly assembl...
3,848280,BEARING 22238 KCAW33C3 BRAND MCB,8482,84,bearing 22238 kcaw33c3 brand mcb bearing ...,bearing_22238 22238_kcaw33c3 kcaw33c3_brand br...
4,630900,USED HANDBAGS AND WALLETS,6309,63,used handbags wallets used handbags wallets us...,used_handbags handbags_wallets wallets_used us...


Sampling function

In [9]:
def bootstrap_sampling(df, test_fraction=0.1, seed=32):
    # Determine the number of test samples
    n_test = int(len(df) * test_fraction)
    # Perform bootstrap sampling for the test set
    test_set = df.sample(n=n_test, replace=True, random_state=seed)
    # Remove the test samples from the original dataframe to create the training set
    train_set = df.drop(test_set.index)
    
    return train_set, test_set

## Iterations definitions

In [10]:
import random
import joblib

fraction = 0.05
iterations = 10

min_val = 0
max_val = 999999999
random_seed = random.randint(min_val, max_val)

seeds = []

for iter in range(iterations):
    seed = random.randint(min_val, max_val)
    seeds.append(seed)

print("Random seeds for each iteration:")
print(seeds)  

out_dir = "results/baselines"
os.makedirs(out_dir, exist_ok=True)

Random seeds for each iteration:
[226944881, 768593320, 366261559, 531210901, 715275432, 908272322, 173892995, 625333247, 370247453, 503070489]


### Doc2Vec

In [11]:
from gensim.models import Doc2Vec

### Evaluating Doc2Vec

Evaluation function

In [12]:
from doc2vec_utils import evaluate_df_d2v

### FastText

FastText configuration to be used as Doc2Vec

In [13]:
from fasttext_utils import FastTextDocVec

### Evaluating FastText

Evaluation function

In [14]:
from fasttext_utils import evaluate_df_ft

In [15]:
# df = df.sample(frac=0.1)

## Training a Doc2Vec as baseline

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

In [16]:
# Model dependences
from gensim.models.doc2vec import TaggedDocument

target_col = 'HS04'
window = 5 # context window size +/- 
num_epochs = 50
model_dim = 254
seed = 32

raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'

#### A- Raw descriptions

In [17]:
all_metrics = []
scored_dfs = {}

text_col = raw_col 

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"D2V_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training Doc2Vec model")
    model = Doc2Vec(window=window, 
                min_count=1, # ignore 1 instead of not ignore any words
                vector_size=model_dim, # dim of the feature vectors
                sample=1e-4, # threshold randomly down-sample high-frequency words
                hs=1, # hierarchical softmax instead of negative sampling
                max_vocab_size=None, # no limit
                alpha=0.025, # initial learning rate
                min_alpha=0.001, # min learning rate
                dm=0, # PV-DBOW 
                dbow_words=0, # only trains doc-vectors
                dm_tag_count=1, # one tag per document
                dm_mean=0, # use the sum of the context word vectors
                dm_concat=0, # for smaller model
                negative=5, # number of negative samples
                seed=seed, # random seed
                workers=os.cpu_count()
                )
    
    print("Building vocabulary")

    sentences = []
    for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
        words_list = row[text_col].split()
        sentences.append(TaggedDocument(words_list, [row[target_col]]))

    print(sentences[:3])

    model.build_vocab(sentences)

    print("Training model")
    model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

    print("Evaluating model")

    df_scored, metrics = evaluate_df_d2v(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")


=== Iteration 1/10 seed 226944881 ===
Model name: D2V_GOODS_DESCRIPTION_HS04_seed226944881
Training Doc2Vec model
Building vocabulary


100%|██████████| 254721/254721 [00:04<00:00, 52329.03it/s]


[TaggedDocument(words=['BRAKE', 'FLUID', 'DOT', '4', '50X200ML'], tags=['2710']), TaggedDocument(words=['LCD', 'ASSEMBLY'], tags=['8443']), TaggedDocument(words=['BEARING', '22238', 'KCAW33C3', 'BRAND', 'MCB'], tags=['8482'])]
Training model
Evaluating model
Model: D2V_GOODS_DESCRIPTION_HS04_seed226944881
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [00:14<00:00, 896.03it/s]


Total samples: 13389
Top-1 Accuracy: 0.5142 (6884/13389)
Top-2 Accuracy: 0.6048 (8098/13389)
Top-3 Accuracy: 0.6496 (8696/13389)
Top-4 Accuracy: 0.6784 (9083/13389)
Top-5 Accuracy: 0.6977 (9340/13389)

=== Iteration 2/10 seed 768593320 ===
Model name: D2V_GOODS_DESCRIPTION_HS04_seed768593320
Training Doc2Vec model
Building vocabulary


...

100%|██████████| 13389/13389 [00:14<00:00, 924.92it/s]


Total samples: 13389
Top-1 Accuracy: 0.5101 (6829/13389)
Top-2 Accuracy: 0.6003 (8037/13389)
Top-3 Accuracy: 0.6462 (8651/13389)
Top-4 Accuracy: 0.6711 (8984/13389)
Top-5 Accuracy: 0.6894 (9231/13389)
=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
D2V_GOODS_DESCRIPTION_HS04_seed173892995,0.519083,0.613339,0.654791,0.682127,0.700799
D2V_GOODS_DESCRIPTION_HS04_seed226944881,0.514228,0.604825,0.649563,0.678393,0.697662
D2V_GOODS_DESCRIPTION_HS04_seed366261559,0.513033,0.605945,0.647920,0.675704,0.694675
D2V_GOODS_DESCRIPTION_HS04_seed370247453,0.510643,0.604601,0.650235,0.677496,0.695422
D2V_GOODS_DESCRIPTION_HS04_seed503070489,0.510120,0.600269,0.646202,0.671073,0.689447
D2V_GOODS_DESCRIPTION_HS04_seed531210901,0.514377,0.602136,0.645605,0.673837,0.692658
D2V_GOODS_DESCRIPTION_HS04_seed625333247,0.501456,0.592352,0.641721,0.666965,0.687654
D2V_GOODS_DESCRIPTION_HS04_seed715275432,0.511913,0.605721,0.647472,0.672567,0.689745
D2V_GOODS_DESCRIPTION_HS04_seed768593320,0.515124,0.609456,0.652028,0.678243,0.694600


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
count,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.512032,0.604354,0.648555,0.675196,0.693719
std,0.004606,0.005552,0.003643,0.004304,0.003984
min,0.501456,0.592352,0.641721,0.666965,0.687654
25%,0.510419,0.602752,0.646520,0.672884,0.690473
50%,0.512473,0.604862,0.648742,0.675629,0.694562
75%,0.514340,0.605889,0.650179,0.678056,0.695235
max,0.519083,0.613339,0.654791,0.682127,0.700799


Saved metrics to results/baselines\scored_dfs_D2V_GOODS_DESCRIPTION_HS04_seed503070489.csv
Saved scored_dfs to results/baselines\scored_dfs_D2V_GOODS_DESCRIPTION_HS04_seed503070489.joblib


In [18]:
#scored_dfs = joblib.load(os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"))

#### B- Preproced descriptions

In [ ]:
all_metrics = []
scored_dfs = {}

text_col = prepro_col 

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"D2V_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training Doc2Vec model")
    model = Doc2Vec(window=window, 
                    min_count=1, # ignore 1 instead of not ignore any words
                    vector_size=model_dim, # dim of the feature vectors
                    sample=1e-4, # threshold randomly down-sample high-frequency words
                    hs=1, # hierarchical softmax instead of negative sampling
                    max_vocab_size=None, # no limit
                    alpha=0.025, # initial learning rate
                    min_alpha=0.001, # min learning rate
                    dm=0, # PV-DBOW 
                    dbow_words=0, # only trains doc-vectors
                    dm_tag_count=1, # one tag per document
                    dm_mean=0, # use the sum of the context word vectors
                    dm_concat=0, # for smaller model
                    negative=5, # number of negative samples
                    seed=seed, # random seed
                    workers=os.cpu_count()
                    )
    
    print("Building vocabulary")

    sentences = []
    for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
        words_list = row[text_col].split()
        sentences.append(TaggedDocument(words_list, [row[target_col]]))

    print(sentences[:3])

    model.build_vocab(sentences)

    print("Training model")
    model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

    print("Evaluating model")

    df_scored, metrics = evaluate_df_d2v(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")


=== Iteration 1/10 seed 226944881 ===
Model name: D2V_PREPRO_DESCRIPTION_HS04_seed226944881
Training Doc2Vec model
Building vocabulary


100%|██████████| 254721/254721 [00:05<00:00, 50779.90it/s]


[TaggedDocument(words=['brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', 'x', 'ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', '4', '50x200ml'], tags=['2710']), TaggedDocument(words=['lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly'], tags=['8443']), TaggedDocument(words=['bearing', '22238', 'kcaw33c3', 'brand', 'mcb', 'bearing', 'kcaw', 'c', 'brand', 'mcb', 'bearing', '22238', 'kcaw33c3', 'brand', 'mcb', 'bearing', '22238', 'kcaw33c3', 'brand', 'mcb'], tags=['8482'])]
Training model
Evaluating model
Model: D2V_PREPRO_DESCRIPTION_HS04_seed226944881
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [00:37<00:00, 359.19it/s]


Total samples: 13389
Top-1 Accuracy: 0.4775 (6393/13389)
Top-2 Accuracy: 0.5591 (7486/13389)
Top-3 Accuracy: 0.5969 (7992/13389)
Top-4 Accuracy: 0.6244 (8359/13389)
Top-5 Accuracy: 0.6427 (8605/13389)

=== Iteration 2/10 seed 768593320 ===
Model name: D2V_PREPRO_DESCRIPTION_HS04_seed768593320
Training Doc2Vec model
Building vocabulary


...

[TaggedDocument(words=['brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', 'x', 'ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', '4', '50x200ml'], tags=['2710']), TaggedDocument(words=['plastic', 'injection', 'mould', 'model', '21a', '110g', 'dsm10102019-16', 'plastic', 'injection', 'mould', 'model', 'a', 'g', 'dsm', 'plastic', 'injection', 'mould', 'model', '21a', '110g', 'dsm10102019', '16', 'plastic', 'injection', 'mould', 'model', '21a', '110g', 'dsm10102019-16'], tags=['8477']), TaggedDocument(words=['lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly'], tags=['8443'])]
Training model
Evaluating model
Model: D2V_PREPRO_DESCRIPTION_HS04_seed715275432
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [00:38<00:00, 351.91it/s]


Total samples: 13389
Top-1 Accuracy: 0.4736 (6341/13389)
Top-2 Accuracy: 0.5569 (7457/13389)
Top-3 Accuracy: 0.5937 (7948/13389)
Top-4 Accuracy: 0.6192 (8290/13389)
Top-5 Accuracy: 0.6376 (8537/13389)

=== Iteration 6/10 seed 908272322 ===
Model name: D2V_PREPRO_DESCRIPTION_HS04_seed908272322
Training Doc2Vec model
Building vocabulary


100%|██████████| 254703/254703 [00:04<00:00, 51283.14it/s]


[TaggedDocument(words=['brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', 'x', 'ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', '4', '50x200ml'], tags=['2710']), TaggedDocument(words=['plastic', 'injection', 'mould', 'model', '21a', '110g', 'dsm10102019-16', 'plastic', 'injection', 'mould', 'model', 'a', 'g', 'dsm', 'plastic', 'injection', 'mould', 'model', '21a', '110g', 'dsm10102019', '16', 'plastic', 'injection', 'mould', 'model', '21a', '110g', 'dsm10102019-16'], tags=['8477']), TaggedDocument(words=['lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly'], tags=['8443'])]
Training model


#### C- Preproced + N-gram descriptions

In [ ]:
all_metrics = []
scored_dfs = {}

text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"D2V_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training Doc2Vec model")
    model = Doc2Vec(window=window, 
                    min_count=1, # ignore 1 instead of not ignore any words
                    vector_size=model_dim, # dim of the feature vectors
                    sample=1e-4, # threshold randomly down-sample high-frequency words
                    hs=1, # hierarchical softmax instead of negative sampling
                    max_vocab_size=None, # no limit
                    alpha=0.025, # initial learning rate
                    min_alpha=0.001, # min learning rate
                    dm=0, # PV-DBOW 
                    dbow_words=0, # only trains doc-vectors
                    dm_tag_count=1, # one tag per document
                    dm_mean=0, # use the sum of the context word vectors
                    dm_concat=0, # for smaller model
                    negative=5, # number of negative samples
                    seed=seed, # random seed
                    workers=os.cpu_count()
                    )
    
    print("Building vocabulary")

    sentences = []
    for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
        words_list = row[text_col].split()
        sentences.append(TaggedDocument(words_list, [row[target_col]]))

    print(sentences[:3])

    model.build_vocab(sentences)

    print("Training model")
    model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

    print("Evaluating model")

    df_scored, metrics = evaluate_df_d2v(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")

=== Iteration 1/3 seed 177566712 ===

Model name: D2V_PREPRO_DESCRIPTIONNGRAM_DESCRIPTION_HS04_seed177566712

Training Doc2Vec model

Building vocabulary


100%|██████████| 25471/25471 [00:00<00:00, 42015.19it/s]


[TaggedDocument(words=['w22', 'sweetheart', 'puff', 'w', 'sweetheart', 'puff', 'w22', 'sweetheart', 'puff', 'w22', 'sweetheart', 'puff', 'w22_sweetheart', 'sweetheart_puff', 'puff_w', 'w_sweetheart', 'sweetheart_puff', 'puff_w22', 'w22_sweetheart', 'sweetheart_puff', 'puff_w22', 'w22_sweetheart', 'sweetheart_puff'], tags=['6206']), TaggedDocument(words=['hand', 'luggage', 'carbon', 'suitcase', 'empty', 'hand', 'luggage', 'carbon', 'suitcase', 'empty', 'hand', 'luggage', 'carbon', 'suitcase', 'empty', 'hand', 'luggage', 'carbon', 'suitcase', 'empty', 'hand_luggage', 'luggage_carbon', 'carbon_suitcase', 'suitcase_empty', 'empty_hand', 'hand_luggage', 'luggage_carbon', 'carbon_suitcase', 'suitcase_empty', 'empty_hand', 'hand_luggage', 'luggage_carbon', 'carbon_suitcase', 'suitcase_empty', 'empty_hand', 'hand_luggage', 'luggage_carbon', 'carbon_suitcase', 'suitcase_empty'], tags=['4202']), TaggedDocument(words=['track', 'link', 'unit', 'track', 'link', 'unit', 'track', 'link', 'unit', 'tra

  0%|          | 0/1338 [00:00<?, ?it/s]C:\Users\santt\AppData\Local\Temp\ipykernel_36912\1009511053.py:37: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  dv = getattr(model, "dv", getattr(model, "docvecs", None))
100%|██████████| 1338/1338 [00:05<00:00, 261.52it/s]


Total samples: 1338
Top-1 Accuracy: 0.4492 (601/1338)
Top-2 Accuracy: 0.5194 (695/1338)
Top-3 Accuracy: 0.5680 (760/1338)
Top-4 Accuracy: 0.5897 (788/1338)
Top-5 Accuracy: 0.6091 (814/1338)
=== Iteration 2/3 seed 61298670 ===

Model name: D2V_PREPRO_DESCRIPTIONNGRAM_DESCRIPTION_HS04_seed61298670

Training Doc2Vec model

Building vocabulary


...

[TaggedDocument(words=['w22', 'sweetheart', 'puff', 'w', 'sweetheart', 'puff', 'w22', 'sweetheart', 'puff', 'w22', 'sweetheart', 'puff', 'w22_sweetheart', 'sweetheart_puff', 'puff_w', 'w_sweetheart', 'sweetheart_puff', 'puff_w22', 'w22_sweetheart', 'sweetheart_puff', 'puff_w22', 'w22_sweetheart', 'sweetheart_puff'], tags=['6206']), TaggedDocument(words=['hand', 'luggage', 'carbon', 'suitcase', 'empty', 'hand', 'luggage', 'carbon', 'suitcase', 'empty', 'hand', 'luggage', 'carbon', 'suitcase', 'empty', 'hand', 'luggage', 'carbon', 'suitcase', 'empty', 'hand_luggage', 'luggage_carbon', 'carbon_suitcase', 'suitcase_empty', 'empty_hand', 'hand_luggage', 'luggage_carbon', 'carbon_suitcase', 'suitcase_empty', 'empty_hand', 'hand_luggage', 'luggage_carbon', 'carbon_suitcase', 'suitcase_empty', 'empty_hand', 'hand_luggage', 'luggage_carbon', 'carbon_suitcase', 'suitcase_empty'], tags=['4202']), TaggedDocument(words=['track', 'link', 'unit', 'track', 'link', 'unit', 'track', 'link', 'unit', 'tra

  0%|          | 0/1338 [00:00<?, ?it/s]C:\Users\santt\AppData\Local\Temp\ipykernel_36912\1009511053.py:37: DeprecationWarning: Call to deprecated `docvecs` (The `docvecs` property has been renamed `dv`.).
  dv = getattr(model, "dv", getattr(model, "docvecs", None))
100%|██████████| 1338/1338 [00:05<00:00, 256.56it/s]


Total samples: 1338
Top-1 Accuracy: 0.4552 (609/1338)
Top-2 Accuracy: 0.5404 (723/1338)
Top-3 Accuracy: 0.5658 (757/1338)
Top-4 Accuracy: 0.5889 (788/1338)
Top-5 Accuracy: 0.6046 (809/1338)

=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
D2V_PREPRO_DESCRIPTIONNGRAM_DESCRIPTION_HS04_seed177566712,0.449178,0.519432,0.568012,0.589686,0.609118
D2V_PREPRO_DESCRIPTIONNGRAM_DESCRIPTION_HS04_seed361568923,0.455157,0.540359,0.565770,0.588939,0.604634
D2V_PREPRO_DESCRIPTIONNGRAM_DESCRIPTION_HS04_seed61298670,0.452915,0.531390,0.568012,0.595665,0.610613


Saved metrics to results/baselines\scored_dfs_D2V_PREPRO_DESCRIPTIONNGRAM_DESCRIPTION_HS04_seed361568923.csv
Saved scored_dfs to results/baselines\scored_dfs_D2V_PREPRO_DESCRIPTIONNGRAM_DESCRIPTION_HS04_seed361568923.joblib


## Training a FastText as baseline

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

In [ ]:
target_col = 'HS04'
window = 5 # context window size +/- 
num_epochs = 50
model_dim = 254
seed = 32

raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'

#### A- Raw descriptions

In [ ]:
all_metrics = []
scored_dfs = {}

text_col = raw_col 

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"FT_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training FastText model")
    model = FastTextDocVec(dim=model_dim, 
                        window=window, 
                        min_count=1, # ignore 1 instead of not ignore any words 
                        epochs=num_epochs, 
                        sg=1, 
                        min_n=3, 
                        max_n=6)

    print("Training model")
    model.fit(train_df, text_col=text_col, label_col=target_col) 
    
    print("Evaluating model")

    df_scored, metrics = evaluate_df_ft(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")

=== Iteration 1/3 seed 177566712 ===

Model name: FT_GOODS_DESCRIPTION_HS04_seed177566712

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_GOODS_DESCRIPTION_HS04_seed177566712
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:17<00:00, 75.62it/s]


Total samples: 1338
Top-1 Accuracy: 0.3543 (473/1338)
Top-2 Accuracy: 0.4537 (606/1338)
Top-3 Accuracy: 0.5112 (684/1338)
Top-4 Accuracy: 0.5411 (723/1338)
Top-5 Accuracy: 0.5620 (752/1338)
=== Iteration 2/3 seed 61298670 ===

Model name: FT_GOODS_DESCRIPTION_HS04_seed61298670

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_GOODS_DESCRIPTION_HS04_seed61298670
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:17<00:00, 75.59it/s]


Total samples: 1338
Top-1 Accuracy: 0.3580 (478/1338)
Top-2 Accuracy: 0.4499 (601/1338)
Top-3 Accuracy: 0.4993 (668/1338)
Top-4 Accuracy: 0.5306 (710/1338)
Top-5 Accuracy: 0.5583 (747/1338)
=== Iteration 3/3 seed 361568923 ===

Model name: FT_GOODS_DESCRIPTION_HS04_seed361568923

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_GOODS_DESCRIPTION_HS04_seed361568923
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:16<00:00, 79.62it/s] 

Total samples: 1338
Top-1 Accuracy: 0.3595 (481/1338)
Top-2 Accuracy: 0.4380 (585/1338)
Top-3 Accuracy: 0.4873 (651/1338)
Top-4 Accuracy: 0.5179 (692/1338)
Top-5 Accuracy: 0.5419 (725/1338)

=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
FT_GOODS_DESCRIPTION_HS04_seed177566712,0.354260,0.453662,0.511211,0.541106,0.562033
FT_GOODS_DESCRIPTION_HS04_seed361568923,0.359492,0.437967,0.487294,0.517937,0.541854
FT_GOODS_DESCRIPTION_HS04_seed61298670,0.357997,0.449925,0.499253,0.530643,0.558296


Saved metrics to results/baselines\scored_dfs_FT_GOODS_DESCRIPTION_HS04_seed361568923.csv
Saved scored_dfs to results/baselines\scored_dfs_FT_GOODS_DESCRIPTION_HS04_seed361568923.joblib


#### B- Preproced descriptions

In [ ]:
all_metrics = []
scored_dfs = {}

text_col = prepro_col 

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"FT_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training FastText model")
    model = FastTextDocVec(dim=model_dim, 
                        window=window, 
                        min_count=1, # ignore 1 instead of not ignore any words 
                        epochs=num_epochs, 
                        sg=1, 
                        min_n=3, 
                        max_n=6)

    print("Training model")
    model.fit(train_df, text_col=text_col, label_col=target_col) 
    
    print("Evaluating model")

    df_scored, metrics = evaluate_df_ft(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")

=== Iteration 1/3 seed 177566712 ===

Model name: FT_PREPRO_DESCRIPTION_HS04_seed177566712

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_PREPRO_DESCRIPTION_HS04_seed177566712
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:17<00:00, 76.55it/s]


Total samples: 1338
Top-1 Accuracy: 0.4410 (590/1338)
Top-2 Accuracy: 0.5561 (744/1338)
Top-3 Accuracy: 0.6181 (827/1338)
Top-4 Accuracy: 0.6577 (879/1338)
Top-5 Accuracy: 0.6854 (916/1338)
=== Iteration 2/3 seed 61298670 ===

Model name: FT_PREPRO_DESCRIPTION_HS04_seed61298670

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_PREPRO_DESCRIPTION_HS04_seed61298670
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:13<00:00, 102.87it/s]


Total samples: 1338
Top-1 Accuracy: 0.4402 (588/1338)
Top-2 Accuracy: 0.5575 (746/1338)
Top-3 Accuracy: 0.6248 (835/1338)
Top-4 Accuracy: 0.6577 (879/1338)
Top-5 Accuracy: 0.6891 (921/1338)
=== Iteration 3/3 seed 361568923 ===

Model name: FT_PREPRO_DESCRIPTION_HS04_seed361568923

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_PREPRO_DESCRIPTION_HS04_seed361568923
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:13<00:00, 100.81it/s]

Total samples: 1338
Top-1 Accuracy: 0.4245 (567/1338)
Top-2 Accuracy: 0.5508 (736/1338)
Top-3 Accuracy: 0.6136 (820/1338)
Top-4 Accuracy: 0.6480 (866/1338)
Top-5 Accuracy: 0.6712 (898/1338)

=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
FT_PREPRO_DESCRIPTION_HS04_seed177566712,0.440957,0.556054,0.618087,0.657698,0.685351
FT_PREPRO_DESCRIPTION_HS04_seed361568923,0.424514,0.550822,0.613602,0.647982,0.671151
FT_PREPRO_DESCRIPTION_HS04_seed61298670,0.440209,0.557549,0.624813,0.657698,0.689088


Saved metrics to results/baselines\scored_dfs_FT_PREPRO_DESCRIPTION_HS04_seed361568923.csv
Saved scored_dfs to results/baselines\scored_dfs_FT_PREPRO_DESCRIPTION_HS04_seed361568923.joblib


#### C- Preproced + N-gram descriptions

In [ ]:
all_metrics = []
scored_dfs = {}

text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"FT_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training FastText model")
    model = FastTextDocVec(dim=model_dim, 
                        window=window, 
                        min_count=1, # ignore 1 instead of not ignore any words 
                        epochs=num_epochs, 
                        sg=1, 
                        min_n=3, 
                        max_n=6)

    print("Training model")
    model.fit(train_df, text_col=text_col, label_col=target_col) 
    
    print("Evaluating model")

    df_scored, metrics = evaluate_df_ft(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")

=== Iteration 1/3 seed 177566712 ===

Model name: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed177566712

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed177566712
Text column: PREPRO_DESCRIPTION_NGRAM_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:16<00:00, 79.50it/s]


Total samples: 1338
Top-1 Accuracy: 0.4290 (574/1338)
Top-2 Accuracy: 0.5643 (754/1338)
Top-3 Accuracy: 0.6173 (825/1338)
Top-4 Accuracy: 0.6472 (866/1338)
Top-5 Accuracy: 0.6741 (902/1338)
=== Iteration 2/3 seed 61298670 ===

Model name: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed61298670

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed61298670
Text column: PREPRO_DESCRIPTION_NGRAM_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:15<00:00, 84.09it/s]


Total samples: 1338
Top-1 Accuracy: 0.4268 (570/1338)
Top-2 Accuracy: 0.5598 (749/1338)
Top-3 Accuracy: 0.6084 (814/1338)
Top-4 Accuracy: 0.6390 (854/1338)
Top-5 Accuracy: 0.6674 (892/1338)
=== Iteration 3/3 seed 361568923 ===

Model name: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed361568923

Training FastText model

Training model

Evaluating model
# Evaluating model: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed361568923
Text column: PREPRO_DESCRIPTION_NGRAM_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 1338/1338 [00:14<00:00, 92.30it/s]

Total samples: 1338
Top-1 Accuracy: 0.4305 (575/1338)
Top-2 Accuracy: 0.5433 (726/1338)
Top-3 Accuracy: 0.5972 (799/1338)
Top-4 Accuracy: 0.6368 (851/1338)
Top-5 Accuracy: 0.6622 (885/1338)

=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed177566712,0.428999,0.564275,0.617339,0.647235,0.674141
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed361568923,0.430493,0.543348,0.597160,0.636771,0.662182
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed61298670,0.426756,0.559791,0.608371,0.639013,0.667414


Saved metrics to results/baselines\scored_dfs_FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed361568923.csv
Saved scored_dfs to results/baselines\scored_dfs_FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed361568923.joblib


## Future steps

- Coding training for baselines using CV or similar method

Applied these iterations to DistiltBERT training and eval